# 04 · Python 고급: 라이브 수집 + 머신러닝  (모듈 4-B)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/04_python_strength.ipynb)

> **STATA로는 어려운** 두 가지를 Python으로: ① World Bank API에서 **실시간 데이터 수집**,
> ② **머신러닝**으로 예측·변수중요도. 수집·자동화·ML은 **Python의 영역**(외부망)이다.

## 1) 라이브 API 수집 — 이 데이터가 만들어진 방식
`wbgapi`로 World Bank에서 직접 가져온다(저장된 CSV가 아니라 라이브). 여러 지표를 한 번에 = 자동화.

In [ ]:
# World Bank 공식 API 패키지 불러오기 (Colab에 없으면 자동 설치)
try:
    import wbgapi as wb
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wbgapi"]); import wbgapi as wb
# 지표코드(GDP·기대수명) × 나라(한국·베트남·케냐·에티오피아) × 연도(2021)를 한 번에 수집
wb.data.DataFrame(["NY.GDP.PCAP.CD","SP.DYN.LE00.IN"], ["KOR","VNM","KEN","ETH"],
                  time=2021, columns="series", labels=False).round(1)

## 2) 머신러닝 — 기대수명 예측 + 변수 중요도
"어떤 개발지표가 기대수명을 가장 잘 예측하나?"를 **랜덤포레스트**로 풀어본다.

- **머신러닝(ML)** = 규칙을 사람이 정하지 않고, 데이터에서 패턴을 *학습*해 예측하는 방법.
- **랜덤포레스트** = 수많은 '결정 트리'를 만들어 평균내는 모델. 비선형·상호작용도 잘 잡는다.
- **왜 train/test로 나누나** = 학습에 쓴 데이터로 평가하면 "외운 것"이라 점수가 부풀려진다.
  *안 본 데이터*로 평가해야 진짜 예측력을 안다(과적합 방지).

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestRegressor      # 랜덤포레스트(여러 결정트리의 앙상블)
from sklearn.linear_model import LinearRegression       # 비교용 선형회귀
from sklearn.model_selection import train_test_split, cross_val_score
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv").dropna(subset=["gdp_pc","life_exp","prim_enroll"]).copy()
df["log_gdp"] = np.log(df["gdp_pc"]); df["log_pop"] = np.log(df["pop"])
# X = 예측에 쓸 입력변수들. 문자형(지역·소득그룹)은 get_dummies로 0/1 더미화(drop_first=중복 제거)
X = pd.get_dummies(df[["log_gdp","log_pop","prim_enroll","region_name","income_name"]], drop_first=True)
y = df["life_exp"]                                       # y = 맞히려는 목표(기대수명)
print("특성:", X.shape[1], "개 · 표본:", len(y))

### train/test 분할 + 교차검증 — '예측력'을 정직하게 평가

In [ ]:
# 데이터를 학습용 75% / 평가용 25%로 분할 (random_state=난수 고정 → 매번 같은 결과)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
rf  = RandomForestRegressor(n_estimators=300, random_state=42).fit(Xtr, ytr)  # 트리 300개로 학습
ols = LinearRegression().fit(Xtr, ytr)                                        # 비교용 선형회귀 학습
# .score = 평가용(학습에 안 쓴) 데이터에서의 R²(예측력). 1에 가까울수록 좋음
print(f"랜덤포레스트  test R² = {rf.score(Xte, yte):.3f}")
print(f"선형회귀(OLS) test R² = {ols.score(Xte, yte):.3f}   ← RF가 비선형을 더 잘 잡음")
print(f"랜덤포레스트  5-fold 교차검증 R² = {cross_val_score(rf, X, y, cv=5).mean():.3f}")  # 5번 나눠 평균(더 보수적)

### 변수 중요도 — 무엇이 기대수명을 가장 잘 예측하나?

In [ ]:
# 변수 중요도 = 랜덤포레스트가 예측에 각 변수를 얼마나 활용했나(클수록 중요). 상위 6개
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(6).round(3)

---
✅ **포인트**: 수집·자동화·머신러닝은 **Python의 영역**(STATA는 사실상 못 함). 입문자도 AI 도움으로
랜덤포레스트·교차검증을 돌린다 — 단, **train/test로 정직하게 평가**하고 결과를 **검증**하는 건 사람 몫.

🧭 **인간 검증**: 변수중요도 1위가 현장 감각과 맞나? '예측을 잘한다'를 *인과(원인)*로 오해하지 않았나?